In [1]:
# =============================================================================
# Projeto: Análise de Recomendações Pedagógicas para o Uso de Inteligência
#          Artificial Generativa na Educação a Distância por meio de PLN e LLMs
# Disciplina: Inteligência Artificial - 7º N - Ciência da Computação
# Universidade Presbiteriana Mackenzie - Prof. Dr. Ivan Carlos Alcântara de Oliveira
# Integrante: Daniel Ribeiro Pauleti - RA 10723162 - 10723162@mackenzista.com.br
#
# Arquivo: notebooks/02_preparacao_dados.ipynb
# Síntese: reconstrução dos resumos (índice invertido), estruturação dos campos,
#          remoção de duplicatas e aplicação dos critérios de inclusão/exclusão.
#
# Histórico de alterações:
#   2026-09-25 - Daniel Ribeiro Pauleti - Versão inicial.
# =============================================================================

# 02 — Preparação dos dados

Etapas: (1) reconstrução dos resumos a partir do índice invertido; (2) estruturação dos campos (título, resumo, palavras-chave, ano, idioma, tipo, DOI, fonte, citações); (3) inferência do idioma quando ausente; (4) remoção de duplicatas (ID, DOI e título normalizado); (5) critérios de inclusão: tipo documental, idioma (en/pt/es), resumo com 50–1000 palavras, menção a IA generativa e a contexto pedagógico, e centralidade do tema.

In [2]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 90)
from preparacao import carregar_bruto, estruturar, aplicar_criterios, reconstruir_resumo
bruto = carregar_bruto()
print("Registros únicos:", len(bruto))

Registros únicos: 471


In [3]:
# Demonstração da reconstrução do resumo
indice = bruto.iloc[0]["abstract_inverted_index"]
print(reconstruir_resumo(indice)[:600], "...")

Generative artificial intelligence (GenAI) has reshaped distance education by prompting a shift towards student-centred initiatives to promote responsible AI usage. This study explores the transformative impact of GenAI in distance learning and focuses on academic integrity and student voices. This study uses the technology acceptance model to investigate how GenAI influences distance education. Three objectives guide the study: (1) exploring the transformative effects of GenAI in distance education, (2) understanding its impact on academic integrity, and (3) examining its influence on student ...


In [4]:
dados = estruturar(bruto)
print("Resumos ausentes:", (dados["n_palavras_resumo"] == 0).sum())
dados["idioma_origem"].value_counts().rename("registros").to_frame()

Resumos ausentes: 52


,registros
idioma_origem,
openalex,311
langdetect,144
ausente,16


In [5]:
corpus, funil = aplicar_criterios(dados)
funil

,etapa,registros,removidos
0,Registros únicos (ID OpenAlex),471,0
1,Remoção de duplicatas por DOI,470,1
2,Remoção de duplicatas por título,424,46
3,"Tipo documental (artigo, revisão, evento, capítulo, preprint)",375,49
4,"Idioma (en, pt, es)",357,18
5,Resumo válido (50 a 1000 palavras),309,48
6,Relevância: menciona IA generativa e contexto pedagógico,307,2
7,Centralidade: IA generativa no título ou >= 2 menções no resumo,208,99


In [6]:
print("Corpus final:", len(corpus), "publicações")
corpus["texto"] = corpus["titulo"] + ". " + corpus["resumo"]
(RAIZ / "data/processed").mkdir(parents=True, exist_ok=True)
corpus.to_csv(RAIZ / "data/processed/corpus.csv", index=False, encoding="utf-8")
funil.to_csv(RAIZ / "data/processed/funil_selecao.csv", index=False, encoding="utf-8")
corpus[["openalex_id", "doi", "ano", "idioma", "tipo", "titulo"]].head(10)

Corpus final: 208 publicações


,openalex_id,doi,ano,idioma,tipo,titulo
0,W4353008813,10.5281/zenodo.7755273,2023,en,article,Challenging the Status Quo and Exploring the New Boundaries in the Age of Algorithms: ...
1,W4367184597,10.5281/zenodo.7857396,2023,en,article,Embracing the future of distance science education: Opportunities and challenges of Ch...
2,W4377094181,10.30935/ojcmt/13291,2023,en,article,ChatGPT: An ever-increasing encroachment of artificial intelligence in online assessme...
3,W4386149392,10.5281/zenodo.8283643,2023,en,article,Revitalizing Online Education: Leveraging Artificial Intelligence Tools for Dynamic an...
4,W4387384844,10.28925/2312-5829.2023.36,2023,en,article,Perspectives of the Use of ChatGPT as a Tool for Online Education of English
5,W4388490750,10.2991/978-94-6463-290-3_3,2023,en,book-chapter,The Empowering of Technology and Innovation for the Sustainability of Intangible Resou...
6,W4388556598,10.18785/jetde.1602.01,2023,en,article,"Generative AI, learning and new literacies"
7,W4388878390,10.1016/j.ssaho.2023.100740,2023,en,article,A review of quality assessment criteria in secondary education with the impact of the ...
8,W4389729898,10.17143/ciaed.xxviiiciaed.2023.231473,2023,pt,conference-paper,O Uso do CHATGPT na Avaliação Individual Online da Disciplina de Projetos Educacionais...
9,W4390099321,10.14742/ajet.8915,2023,en,article,Investigation of student experiences with ChatGPT-supported online learning applicatio...


In [7]:
# Verificação de qualidade do corpus final
print("Duplicatas por DOI:", corpus["doi"].dropna().duplicated().sum())
print("Registros sem DOI:", corpus["doi"].isna().sum())
print("Valores ausentes por coluna:")
corpus.isna().sum()[lambda s: s > 0]

Duplicatas por DOI: 0
Registros sem DOI: 6
Valores ausentes por coluna:


doi                  6
topico_principal     3
area                 3
fonte               38
dtype: int64